HASTS206 Assignment 1 – Group 3
Household access to clean cooking energy in Zimbabwe: Python analysis of the digital pilot
Scope. This notebook cleans and analyses the raw KoBoToolbox export from a pilot of 34 simulated household responses (urban and rural). The pilot tests the proposed workflow (form → Python → Power BI). Its results describe the pilot records only and cannot establish national prevalence.

Source label. Simulated household responses submitted through the actual KoBoToolbox form by the five enumerators. No names, phone numbers, precise addresses or account details were collected.

How to rerun. Open this notebook from the repository (the notebooks/ folder or the repository root), then Kernel → Restart & Run All. Paths are relative; the raw export in data/raw/ is read and never modified. Outputs are written to outputs/.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ==========================================
# 1. SETUP PATHS AND CONSTANTS
# ==========================================
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / "data" / "raw").exists() else HERE.parent
RAW_DIR = ROOT / "data" / "raw"
OUT = ROOT / "outputs"
FIG = OUT / "figures"
for p in (OUT, FIG):
    p.mkdir(parents=True, exist_ok=True)

# Load the raw file
RAW_FILE = sorted(RAW_DIR.glob("Zim_cooking_fuels*.csv"))[0]
print(f"Reading: {RAW_FILE.name}")

CLEAN_FUELS = {"electricity", "gas", "solar"}
STOVE_ELIGIBLE_FUELS = {"firewood", "charcoal"}
SOLID_NOT_ROUTED = {"dung_or_crop_residue"}
VALID_FUELS = CLEAN_FUELS | STOVE_ELIGIBLE_FUELS | SOLID_NOT_ROUTED | {"paraffin"}
UNKNOWN_CODE, REFUSED_CODE = "dont_know", "refused"
EXP_MIN, EXP_MAX = 0, 500
DOMAINS = ["urban", "rural"]
tcrit = 2.228 # Used for weighting demo

# ==========================================
# 2. LOAD AND CLEAN DATA
# ==========================================
raw = pd.read_csv(RAW_FILE, sep=";", dtype=str)
work = raw.copy()

# Basic cleaning
d = work.copy()
d["source"] = "simulated household response via KoBoToolbox form (pilot)"
d["submitted_at"] = pd.to_datetime(d["_submission_time"])
d["submitted_date"] = d["submitted_at"].dt.strftime("%Y-%m-%d")
order = d.sort_values("submitted_at")["__version__"].drop_duplicates().tolist()
d["form_version"] = d["__version__"].map({v: f"v{i + 1}" for i, v in enumerate(order)})
d["interview_outcome"] = d["Record_Interview_outcome"]

# Numeric conversion
d["fuel_expenditure_usd"] = pd.to_numeric(d["fuel_expenditure_usd"], errors="coerce")
bad_range = d["fuel_expenditure_usd"].notna() & ~d["fuel_expenditure_usd"].between(EXP_MIN, EXP_MAX)
d.loc[bad_range, "fuel_expenditure_usd"] = np.nan

# Indicator 1
d["fuel_status"] = np.select([d["primary_fuel"] == UNKNOWN_CODE, d["primary_fuel"] == REFUSED_CODE],
                             ["unknown", "refused"], default="observed")
d["fuel_group"] = np.select(
    [d["primary_fuel"].isin(CLEAN_FUELS), d["primary_fuel"].isin(STOVE_ELIGIBLE_FUELS),
     d["primary_fuel"].isin(SOLID_NOT_ROUTED), d["primary_fuel"] == "paraffin"],
    ["clean", "solid (firewood/charcoal)", "solid (dung/crop residue)", "paraffin"], default="unknown/refused")
d["ind1_den"] = (d["fuel_status"] == "observed").astype(int)
d["ind1_num"] = (d["ind1_den"].eq(1) & d["primary_fuel"].isin(CLEAN_FUELS)).astype(int)

# Indicator 2
d["stove_eligible"] = d["primary_fuel"].isin(STOVE_ELIGIBLE_FUELS).astype(int)
d["stove_status"] = np.select(
    [d["stove_eligible"].eq(1) & d["has_improved_stove"].isin(["yes", "no"]),
     d["stove_eligible"].eq(1) & (d["has_improved_stove"] == REFUSED_CODE),
     d["stove_eligible"].eq(1) & (d["has_improved_stove"] == UNKNOWN_CODE),
     d["stove_eligible"].eq(1),
     d["primary_fuel"].isin(SOLID_NOT_ROUTED)],
    ["observed", "refused", "unknown", "missing_unexpected", "not_routed"], default="inapplicable")
d["ind2_den"] = (d["stove_status"] == "observed").astype(int)
d["ind2_num"] = (d["ind2_den"].eq(1) & (d["has_improved_stove"] == "yes")).astype(int)

# Indicator 3
d["exp_status"] = np.select(
    [(d["fuel_purchased"] == "yes") & d["fuel_expenditure_usd"].notna(),
     (d["fuel_purchased"] == "no_0") & (d["fuel_expenditure_usd"] == 0),
     d["fuel_purchased"] == UNKNOWN_CODE, d["fuel_purchased"] == REFUSED_CODE],
    ["observed_purchased", "observed_zero_not_purchased", "unknown", "refused"], default="invalid_or_missing")
d["ind3_den"] = d["exp_status"].isin(["observed_purchased", "observed_zero_not_purchased"]).astype(int)
d["exp_value_valid"] = d["fuel_expenditure_usd"].where(d["ind3_den"].eq(1))

# Final case file
case_cols = ["case_id", "domain", "source", "form_version", "submitted_date", "interview_outcome",
             "primary_fuel", "fuel_group", "fuel_status", "ind1_den", "ind1_num",
             "has_improved_stove", "stove_used_last7", "stove_eligible", "stove_status", "ind2_den", "ind2_num",
             "fuel_purchased", "fuel_expenditure_usd", "exp_status", "ind3_den", "exp_value_valid"]
cases = d[case_cols].sort_values("case_id").reset_index(drop=True)

# ==========================================
# 3. GENERATE BENCHMARK TABLES
# ==========================================
def indicator_table(frame, label):
    rows = []
    n = len(frame)
    den, num = int(frame["ind1_den"].sum()), int(frame["ind1_num"].sum())
    rows.append(dict(indicator_id="I1", indicator="Clean fuel use", domain=label, numerator=num,
                     denominator=den, value=round(100 * num / den, 1) if den else np.nan, unit="% of valid households",
                     records_in_domain=n, excluded_from_denominator=n - den))
    den, num = int(frame["ind2_den"].sum()), int(frame["ind2_num"].sum())
    rows.append(dict(indicator_id="I2", indicator="Improved stove ownership", domain=label,
                     numerator=num, denominator=den, value=round(100 * num / den, 1) if den else np.nan,
                     unit="% of eligible households", records_in_domain=n, excluded_from_denominator=n - den))
    v = frame["exp_value_valid"].dropna()
    rows.append(dict(indicator_id="I3", indicator="Mean monthly cooking-fuel expenditure", domain=label,
                     numerator=round(v.sum(), 2), denominator=int(len(v)), value=round(v.mean(), 2) if len(v) else np.nan,
                     unit="USD per household per month", records_in_domain=n, excluded_from_denominator=n - len(v)))
    return rows

rows = indicator_table(cases, "Overall")
for dom in DOMAINS:
    rows += indicator_table(cases[cases["domain"] == dom], dom)
benchmark_detail = pd.DataFrame(rows)
benchmark = benchmark_detail[["indicator_id", "indicator", "domain", "numerator", "denominator", "value", "unit"]]

# Missingness Summary
def status_table(col, name):
    t = pd.crosstab(cases[col], cases["domain"]).reindex(columns=DOMAINS, fill_value=0)
    t["Total"] = t.sum(axis=1)
    t.insert(0, "indicator", name)
    return t.reset_index().rename(columns={col: "status"})

missingness = pd.concat([
    status_table("fuel_status", "I1 primary_fuel"),
    status_table("stove_status", "I2 improved stove"),
    status_table("exp_status", "I3 expenditure")], ignore_index=True)

# ==========================================
# 4. GENERATE CHARTS
# ==========================================
# Chart 1
t1 = benchmark[(benchmark.indicator_id == "I1") & (benchmark.domain.isin(DOMAINS))].set_index("domain").loc[DOMAINS]
fig, ax = plt.subplots(figsize=(7.5, 4.8))
bars = ax.bar(["Urban", "Rural"], t1["value"], color=["#2c7fb8", "#7fcdbb"], width=0.55)
for b, (_, r) in zip(bars, t1.iterrows()):
    ax.text(b.get_x() + b.get_width() / 2, r["value"] + 2, f"{r['value']:.1f}%\n({int(r['numerator'])}/{int(r['denominator'])})",
            ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_ylim(0, 118)
ax.set_ylabel("Households using clean fuel (%)")
ax.set_title("Clean primary cooking fuel: urban vs rural", fontsize=12)
all_row = benchmark[(benchmark.indicator_id == "I1") & (benchmark.domain == "Overall")].iloc[0]
ax.axhline(all_row["value"], color="grey", ls="--", lw=1)
ax.set_xlim(-0.5, 1.5)
fig.tight_layout()
fig.savefig(FIG / "fig1_clean_fuel_by_domain.png", dpi=200)
plt.close(fig)

# Chart 2
fuel_order = ["electricity", "gas", "solar", "paraffin", "firewood", "charcoal", "dung_or_crop_residue", "refused"]
colors = ["#08519c", "#3182bd", "#9ecae1", "#fdae6b", "#8c510a", "#bf812d", "#dfc27d", "#bdbdbd"]
ct = pd.crosstab(cases["domain"], cases["primary_fuel"]).reindex(index=DOMAINS, columns=fuel_order, fill_value=0)
fig, ax = plt.subplots(figsize=(9, 3.9))
left = np.zeros(len(ct))
for f, c in zip(fuel_order, colors):
    ax.barh(["Urban", "Rural"], ct[f], left=left, color=c, label=f.replace("_", " "), edgecolor="white")
    left += ct[f].values
ax.set_xlabel("Number of pilot households")
ax.set_title("Primary cooking fuel mix by domain", fontsize=12)
ax.legend(ncol=4, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.25), frameon=False)
fig.tight_layout()
fig.savefig(FIG / "fig2_fuel_mix_by_domain.png", dpi=200)
plt.close(fig)

# Chart 3
fig, ax = plt.subplots(figsize=(7.5, 4.8))
rng = np.random.default_rng(1)
for i, dom in enumerate(DOMAINS):
    v = cases.loc[cases["domain"] == dom, "exp_value_valid"].dropna()
    ax.scatter(i + rng.uniform(-0.12, 0.12, len(v)), v, s=40, alpha=0.75, color=["#2c7fb8", "#7fcdbb"][i], edgecolor="k", lw=0.4)
    ax.hlines(v.mean(), i - 0.3, i + 0.3, color="red", lw=2)
ax.set_xticks([0, 1]); ax.set_xticklabels(["Urban", "Rural"])
ax.set_ylabel("Monthly cooking-fuel expenditure (USD)")
ax.set_title("Monthly cooking-fuel expenditure", fontsize=12)
fig.tight_layout()
fig.savefig(FIG / "fig3_expenditure_by_domain.png", dpi=200)
plt.close(fig)

# ==========================================
# 5. EXPORT ALL CSV FILES FOR POWER BI
# ==========================================
cases.to_csv(OUT / "clean_case_level.csv", index=False)
benchmark.to_csv(OUT / "benchmark.csv", index=False)
benchmark_detail.to_csv(OUT / "benchmark_detail.csv", index=False)
missingness.to_csv(OUT / "missingness_summary.csv", index=False)

# Data dictionary
dictionary = pd.DataFrame([
    ("case_id", "Unique case ID, format ZIM-0001"),
    ("domain", "Reporting domain: urban / rural"),
    ("source", "Provenance label: simulated household response (pilot)"),
    ("form_version", "KoBoToolbox form version (v1, v2)"),
    ("submitted_date", "Date of submission"),
    ("interview_outcome", "complete / partial"),
    ("primary_fuel", "Main cooking fuel"),
    ("fuel_group", "Derived grouping of primary_fuel"),
    ("fuel_status", "observed / unknown / refused (Indicator 1)"),
    ("ind1_den", "1 = in Indicator 1 denominator"),
    ("ind1_num", "1 = in Indicator 1 numerator (clean fuel)"),
    ("has_improved_stove", "Improved cookstove ownership"),
    ("stove_used_last7", "Improved stove used in last 7 days"),
    ("stove_eligible", "1 = firewood/charcoal user"),
    ("stove_status", "observed / refused / unknown / not_routed / inapplicable"),
    ("ind2_den", "1 = eligible AND valid yes/no answer"),
    ("ind2_num", "1 = in denominator AND stove = yes"),
    ("fuel_purchased", "Fuel bought last month"),
    ("fuel_expenditure_usd", "Monthly expenditure in USD"),
    ("exp_status", "observed_purchased / observed_zero_not_purchased / unknown / refused"),
    ("ind3_den", "1 = valid expenditure observation"),
    ("exp_value_valid", "Expenditure used in Indicator 3"),
], columns=["field", "description"])
dictionary.to_csv(OUT / "data_dictionary_clean_case_level.csv", index=False)

# Template for Power BI reconciliation
template = benchmark[["indicator_id", "domain", "numerator", "denominator", "value"]].rename(
    columns={"numerator": "python_numerator", "denominator": "python_denominator", "value": "python_value"})
template[["powerbi_numerator", "powerbi_denominator", "powerbi_value"]] = np.nan
template.to_csv(OUT / "powerbi_values_TEMPLATE.csv", index=False)

print("\n✅ SUCCESS! All files have been saved to the 'outputs/' folder.")
print("Open 'outputs/clean_case_level.csv' in Power BI to build your visualizations.")

Reading: Zim_cooking_fuels_-_all_versions_-_False_-_2026-11-12-06-31-58.csv

✅ SUCCESS! All files have been saved to the 'outputs/' folder.
Open 'outputs/clean_case_level.csv' in Power BI to build your visualizations.
